# Fine-tune Homer ACT corrections on Kaggle
Upload `act-correction-training.zip` as a **private Kaggle Dataset**, attach it to this notebook, choose a GPU accelerator, and enable Internet. Then select **Save Version → Save & Run All** to persist results. This continues from the included previous Kaggle policy for 2,000 steps, mixing the 56 original demonstrations with 18 hand corrections (50/50 sampling). Both cameras, six joints and the saved normalization are preserved. It does not operate the robot.

Checkpoints and logs go to `/kaggle/working/act-correction-output`. Download `act-trained-results.zip` when finished. The resumable optimizer is retained in `act-correction-output/latest`; it is not included in the smaller return ZIP. Re-running within the same session resumes automatically. Across sessions, attach the previous notebook's saved output as an input to restore its latest checkpoint. Keep the dataset unchanged when resuming.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile

WORK = Path('/kaggle/working')
ROOT = WORK / 'act-correction-training'
OUTPUT = WORK / 'act-correction-output'
archives = sorted(Path('/kaggle/input').rglob('act-correction-training.zip'))
if not (ROOT / 'dataset.json').exists():
    if archives:
        with zipfile.ZipFile(archives[0]) as archive:
            archive.extractall(WORK)
    else:
        # Kaggle may unpack a dataset ZIP automatically.
        candidates = sorted(Path('/kaggle/input').rglob('dataset.json'))
        sources = [p.parent for p in candidates if (p.parent / 'train.py').exists() and (p.parent / 'data').is_dir() and (p.parent / 'initial-policy').is_dir()]
        if not sources:
            raise FileNotFoundError('Attach the private dataset containing act-correction-training.zip.')
        shutil.copytree(sources[0], ROOT, dirs_exist_ok=True)
manifest = json.loads((ROOT / 'dataset.json').read_text())
print('Dataset:', len(manifest['train']), 'train demos;', len(manifest['validation']), 'validation demos')

# Restore saved notebook output when resuming in a new session.
if not (OUTPUT / 'latest' / 'training.pt').exists():
    saved = sorted(Path('/kaggle/input').rglob('training.pt'))
    saved = [p for p in saved if p.parent.name == 'latest' and (p.parent.parent / 'dataset.json').exists()
             and json.loads((p.parent.parent / 'dataset.json').read_text()).get('correction_train')
             == manifest['correction_train']
             and json.loads((p.parent.parent / 'dataset.json').read_text()).get('train') == manifest['train']
             and json.loads((p.parent.parent / 'dataset.json').read_text()).get('validation') == manifest['validation']]
    if saved:
        print('Restoring:', saved[0].parent.parent)
        shutil.copytree(saved[0].parent.parent, OUTPUT, dirs_exist_ok=True)


In [ ]:
# Separate Python 3.11: independent of Kaggle's kernel and ensurepip.
ENV = Path('/tmp/act-kaggle-py311')
PYTHON = ENV / 'bin/python'
def run(command):
    subprocess.run([str(x) for x in command], check=True)

if not PYTHON.exists():
    tools_dir = Path('/tmp/act-kaggle-tools')
    run([sys.executable, '-m', 'pip', 'install', '--target', tools_dir, 'uv'])
    uv_env = dict(os.environ, PYTHONPATH=str(tools_dir))
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.11',
                    '--managed-python', '--seed', str(ENV)], env=uv_env, check=True)
if not (ENV / 'ready').exists():
    run([PYTHON, '-m', 'pip', 'install', '--upgrade', 'pip'])
    run([PYTHON, '-m', 'pip', 'install', 'torch==2.7.1', 'torchvision==0.22.1',
         '--index-url', 'https://download.pytorch.org/whl/cu126'])
    run([PYTHON, '-m', 'pip', 'install', '-r', ROOT / 'requirements.txt'])
    (ENV / 'ready').touch()
run([PYTHON, '-c', "import torch; assert torch.cuda.is_available(), 'Select a GPU accelerator in Kaggle settings'; print('GPU:', torch.cuda.get_device_name(0), 'Torch:', torch.__version__)"])


In [ ]:
STEPS = 2000
BATCH_SIZE = 8  # Uses one GPU, including when Kaggle supplies two T4s.
OUTPUT.mkdir(exist_ok=True)
command = [PYTHON, '-u', ROOT / 'train.py', '--data', ROOT / 'data',
           '--dataset-manifest', ROOT / 'dataset.json', '--learning-rate', '0.00001', '--output', OUTPUT,
           '--device', 'cuda', '--batch-size', BATCH_SIZE, '--threads', 2,
           '--steps', STEPS, '--save-every', 200]
previous = OUTPUT / 'checkpoint-previous'
if not (OUTPUT / 'latest').exists() and (previous / 'training.pt').exists():
    previous.rename(OUTPUT / 'latest')
if (OUTPUT / 'latest' / 'training.pt').exists():
    command.append('--resume')
else:
    command.extend(['--init-from', ROOT / 'initial-policy'])
with (OUTPUT / 'terminal.log').open('a', buffering=1) as log:
    process = subprocess.Popen([str(x) for x in command], stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, start_new_session=True)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
        status = process.wait()
    except KeyboardInterrupt:
        import signal
        process.send_signal(signal.SIGINT)
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
        process.wait()
        raise
if status:
    raise RuntimeError(f'Training exited with status {status}; inspect terminal.log. Rerun to resume.')


In [ ]:
checkpoint = OUTPUT / ('best' if (OUTPUT / 'best').exists() else 'latest')
destination = WORK / 'act-trained-results.zip'
with zipfile.ZipFile(destination, 'w', zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
    for path in sorted(checkpoint.rglob('*')):
        if path.is_file() and path.name != 'training.pt':
            archive.write(path, 'policy/' + path.relative_to(checkpoint).as_posix())
    for name in ('dataset.json', 'run.json', 'metrics.jsonl', 'terminal.log'):
        if (OUTPUT / name).exists():
            archive.write(OUTPUT / name, name)
    archive.writestr('README.txt', 'ACT fine-tuned from the previous Kaggle policy on original demonstrations and hand corrections. Inputs: wrist and overview RGB plus six raw joint ticks. Use saved pre/post processors. Offline validation does not establish physical grasp success.\n')
print('Download:', destination)
print('Best validation:', (checkpoint / 'validation.json').read_text() if (checkpoint / 'validation.json').exists() else 'not available')
